# Baseline Logistic Regression + TF-IDF

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn import over_sampling
from IPython.display import display

# Reading product review sentiment file
df_prod_review = pd.read_csv('https://raw.githubusercontent.com/aakashgoel12/blogs/master/input/product_review_sentiment.csv',\
                      encoding='latin-1')
display(df_prod_review.sample(n=5, random_state=42))

,Review,user_sentiment
9329,fresh clean smell everything need quick clean ...,1
4160,great vacuum love lightweight vacuum easy carr...,1
18500,smell great wipe easy use work smell great,1
8840,product count use clorox wipe everything trave...,1
5098,great movie excellent movie add blu ray collec...,1


In [ ]:
x=df_prod_review['Review']
y=df_prod_review['user_sentiment']
print("Distribusi sentimen positif dan negatif dalam ulasan: \n{}".format(y.value_counts(normalize=True)))
# Split the dataset into test and train
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=50)

# As we saw above that data is imbalanced, balance training data using over sampling

ros = over_sampling.RandomOverSampler(random_state=0)
X_train, y_train = ros.fit_resample(pd.DataFrame(X_train), pd.Series(y_train))
print("Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: \n{}".format(y_train.value_counts(normalize=True)))
# Convert into list of string
X_train = X_train['Review'].tolist()

Distribusi sentimen positif dan negatif dalam ulasan: 
user_sentiment
1    0.888401
0    0.111599
Name: proportion, dtype: float64
Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: 
user_sentiment
1    0.5
0    0.5
Name: proportion, dtype: float64


In [ ]:
word_vectorizer = TfidfVectorizer(strip_accents='unicode', token_pattern=r'\w{1,}',\
                                ngram_range=(1, 3), stop_words='english', sublinear_tf=True)

# Fiting it on Train
word_vectorizer.fit(X_train)
# Transforming the train and test datasets
X_train_transformed = word_vectorizer.transform(X_train)
X_test_transformed = word_vectorizer.transform(X_test.tolist())

In [ ]:
def evaluate_model(y_pred,y_actual):
    print(classification_report(y_true = y_actual, y_pred = y_pred))
    # Confusion matrix
    cm = confusion_matrix(y_true = y_actual, y_pred = y_pred)
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]
    # Calculating the Sensitivity
    sensitivity = round(TP/float(FN + TP),2)
    print("sensitivity: {}".format(sensitivity))
    # Calculating the Specificity
    specificity = round(TN / float(TN + FP),2)
    print("specificity: {}".format(specificity))

# Model Training
lr_model = LogisticRegression()
lr_model.fit(X_train_transformed,y_train)
# Prediction on Train Data
y_pred_train= lr_model.predict(X_train_transformed)
# Prediction on Test Data
y_pred_test = lr_model.predict(X_test_transformed)
# Evaluation on Train
print("Evaluation on Train dataset ..")
evaluate_model(y_pred = y_pred_train, y_actual = y_train)
print("Evaluation on Test dataset ..")
# Evaluation on Test
evaluate_model(y_pred = y_pred_test, y_actual = y_test)

Evaluation on Train dataset ..
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     16734
           1       1.00      0.98      0.99     16734

    accuracy                           0.99     33468
   macro avg       0.99      0.99      0.99     33468
weighted avg       0.99      0.99      0.99     33468

sensitivity: 0.98
specificity: 1.0
Evaluation on Test dataset ..
              precision    recall  f1-score   support

           0       0.53      0.58      0.56       922
           1       0.95      0.93      0.94      7140

    accuracy                           0.89      8062
   macro avg       0.74      0.76      0.75      8062
weighted avg       0.90      0.89      0.90      8062

sensitivity: 0.93
specificity: 0.58


##Error Analysis

In [ ]:
true_labels = y_test.tolist()
pred_labels = y_pred_test.tolist()
texts = X_test.tolist()

error_df = pd.DataFrame({
    'text': texts,
    'actual': true_labels,
    'predicted': pred_labels
})

In [ ]:
# Save actual vs predicted results for all data

display(error_df.sample(20, random_state=42))
error_df.to_csv('prediksi vs actual LR+TF-IDF.csv', index=False)

,text,actual,predicted
742,awsome product really good cleaning defect rea...,1,1
2126,refresh year old daughter love lip shimmer ...,1,1
3689,great story didnt purchase self however person...,1,1
4586,worth every cent product best moisturizer lips...,1,1
6289,good movie good,1,1
1302,germ free love cleaning minute feel nice germ ...,1,1
1168,daughter love movie silly movie cute daughter ...,1,1
7139,fun movie family fun watch together good humor...,1,1
3049,favortie clorox wipe favorite clean everyday r...,1,1
2778,vampire life hard sometimes bold humor movie a...,1,1


In [ ]:
# Inspect misclassified samples
misclassified = error_df[error_df['actual'] != error_df['predicted']]

print("Jumlah total error:", len(misclassified))
display(misclassified.sample(10, random_state=42))

Jumlah total error: 857


,text,actual,predicted
6772,love product one buy product trust aveeno bran...,0,1
5729,favorite lip color use product time replace li...,0,1
1035,one favorite lip color little pigment lip colo...,0,1
1777,great use clorox home work safe convenient kee...,1,0
3582,expect tried conditioner along shampoo expect ...,0,1
7465,best thing since slice bread live pool never g...,0,1
6733,fantastic excellent product use especially col...,1,0
2294,easy effective love convenience effectiveness ...,0,1
6589,one star one person family like taste,1,0
7782,bad movie watch youre bore good part also bore...,1,0


##Sampling

In [ ]:
sampled_df = error_df.sample(n=300, random_state=42)
display(sampled_df)

,text,actual,predicted
742,awsome product really good cleaning defect rea...,1,1
2126,refresh year old daughter love lip shimmer ...,1,1
3689,great story didnt purchase self however person...,1,1
4586,worth every cent product best moisturizer lips...,1,1
6289,good movie good,1,1
...,...,...,...
5019,change good thing use exxtra gel long remember...,0,0
7450,clorox wipe love clorox product especially wip...,1,1
1606,great product smooth go easily immediately abs...,1,1
1405,great product great product use daily housethe...,1,1


In [ ]:
# Inspect misclassified samples
misclassified_sample = sampled_df[sampled_df['actual'] != sampled_df['predicted']]

print("Jumlah total error:", len(misclassified_sample))
display(misclassified_sample.sample(10, random_state=42))

misclassified.to_csv('misclassified.csv', index=False)

Jumlah total error: 25


,text,actual,predicted
1203,awesome film would recommend film anyone fan o...,0,1
1053,cheetos grandson love go bag every couple day let,0,1
5075,easy use shock change like product doesnt chan...,0,1
932,wipe hand never many clorox wipe hand family n...,1,0
1009,disappoint enjoy st movie must watch lot laugh,1,0
1617,please bring back old formula happy old formul...,1,0
676,love product would recommend family friend,0,1
7886,great organizational product folder assist org...,0,1
1992,best wipe love smell cant go wrong clorox prod...,0,1
1452,surprised vacuum clean read review hesitant pu...,0,1


In [ ]:
# Create a copy before adding baseline prediction labels
sampled_with_acc = sampled_df.copy()

# acc_a: 1 = correct prediction, 0 = misclassified
sampled_with_acc['acc_a'] = sampled_with_acc['text'].apply(
    lambda x: 0 if x in misclassified_sample['text'].values else 1
)

display(sampled_with_acc)

,text,actual,predicted,acc_a
742,awsome product really good cleaning defect rea...,1,1,1
2126,refresh year old daughter love lip shimmer ...,1,1,1
3689,great story didnt purchase self however person...,1,1,1
4586,worth every cent product best moisturizer lips...,1,1,1
6289,good movie good,1,1,1
...,...,...,...,...
5019,change good thing use exxtra gel long remember...,0,0,1
7450,clorox wipe love clorox product especially wip...,1,1,1
1606,great product smooth go easily immediately abs...,1,1,1
1405,great product great product use daily housethe...,1,1,1


In [ ]:
sampled_with_acc.to_csv('LR_sampel.csv', index=False)